# ShopFlow — Phase 1 COPY INTO
### Move data from stage into all 9 RAW tables — pure SQL

---

### Before running this notebook

**How to upload the downloaded CSVs to the stage:**
1. Download the Olist dataset from Kaggle to your local machine and unzip it.
2. In the Snowflake Snowsight UI, navigate to: **Data → Databases → SHOPFLOW_DB → SHOPFLOW_RAW → Stages → OLIST_STAGE**
3. Click the **+ Files** button in the top right corner.
4. Select all 9 unzipped CSV files from your local machine and click **Upload**. Snowflake will automatically compress them to `.gz`.

---

Confirm all 9 CSV files have been uploaded to the stage via Snowsight UI.

You should have these files in the stage:

| File in stage | Loads into |
|---------------|------------|
| olist_orders_dataset.csv.gz | RAW_ORDERS |
| olist_order_items_dataset.csv.gz | RAW_ORDER_ITEMS |
| olist_customers_dataset.csv.gz | RAW_CUSTOMERS |
| olist_sellers_dataset.csv.gz | RAW_SELLERS |
| olist_products_dataset.csv.gz | RAW_PRODUCTS |
| olist_order_payments_dataset.csv.gz | RAW_PAYMENTS |
| olist_order_reviews_dataset.csv.gz | RAW_REVIEWS |
| olist_geolocation_dataset.csv.gz | RAW_GEOLOCATION |
| product_category_name_translation.csv.gz | RAW_CATEGORY_TRANSLATION |

Files are stored as `.csv.gz` because Snowflake auto-compresses them on upload.

---

### Run cells one at a time — top to bottom

Each cell contains exactly one SQL statement. This way every result is visible directly below the cell that produced it.

---

### Key Snowflake concepts in this notebook

| Topic | Where you see it |
|-------|------------------|
| COPY INTO syntax | Every load cell |
| FILE_FORMAT — FORMAT_NAME | Every load cell |
| ON_ERROR options | Every load cell |
| PURGE option | Every load cell |
| COPY INTO idempotency | Cell 2 explanation |
| LIST stage | Cell 2 |
| LOAD_HISTORY | Cell 14 |
| INFORMATION_SCHEMA.TABLES | Cell 13 |

---
## Cell 1 — Set session context

Before running any SQL, tell Snowflake which warehouse, database, and schema to use.

### Why this matters

- `USE WAREHOUSE` — activates the compute engine. Without this, SQL statements queue but nothing executes.
- `USE DATABASE` + `USE SCHEMA` — sets the default namespace. This means we can write `RAW_ORDERS` instead of the fully qualified `SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDERS` in every statement.

The `SELECT CURRENT_*` at the end confirms all three are set correctly before you proceed.

> **💡 Pro Tip: Context is King**
> 
> If you reference an object without a fully qualified name and the wrong schema is active, Snowflake will throw `Object does not exist`. Always verify context at the top of every script.

In [ ]:
--USE WAREHOUSE SHOPFLOW_WH;
USE DATABASE  SHOPFLOW_DB;
USE SCHEMA    SHOPFLOW_RAW;

In [ ]:
-- Verify context — all three should show SHOPFLOW values
SELECT
    CURRENT_WAREHOUSE() AS active_warehouse,
    CURRENT_DATABASE()  AS active_database,
    CURRENT_SCHEMA()    AS active_schema,
    CURRENT_ROLE()      AS active_role;

---
## Cell 2 — Confirm files are in the stage

`LIST @stage_name` shows every file currently sitting in a stage — names, sizes, and last modified timestamps.

Run this before any COPY INTO. If you see fewer than 9 files, go back to Snowsight and upload the missing ones first.

### What the .gz extension means

When you upload a CSV via the Snowsight UI, Snowflake automatically compresses it to `.gz`. So `olist_orders_dataset.csv` becomes `olist_orders_dataset.csv.gz` in the stage. This is why every `COPY INTO` below references the `.gz` filename.

> **🛡️ Reliability Tip: COPY INTO Idempotency**
> 
> Snowflake tracks which files have already been loaded into each table. If you run a `COPY INTO` on a file that was already loaded, it returns status `SKIPPED` — it will **not** load the data again and will **not** create duplicate rows. To force a reload anyway, add `FORCE = TRUE` to the statement.

In [ ]:
-- List all files currently in the stage
-- You should see 9 files, all ending in .csv.gz
-- If fewer than 9 appear, upload the missing files before continuing
LIST @SHOPFLOW_DB.SHOPFLOW_RAW.OLIST_STAGE;

---
## Cells 3 – 11 — COPY INTO each RAW table

### COPY INTO syntax explained once — applies to all 9 cells

```sql
COPY INTO target_table                    -- which table to load data into
FROM @stage/filename.csv                  -- which file in the stage to read from
                                          -- note: no .gz — files uploaded via
                                          -- Snowsight UI are not compressed
FILE_FORMAT = (
    FORMAT_NAME                    = 'OLIST_CSV_FORMAT',
                                          -- references the named file format created
                                          -- in the setup notebook — defines delimiter,
                                          -- quotes, NULLs, encoding in one place
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
                                          -- RAW tables have 2 extra metadata columns
                                          -- (_loaded_at, _source_file) that don't exist
                                          -- in the CSV — FALSE tells Snowflake to load
                                          -- what's there and use DEFAULT for the rest
)
ON_ERROR = 'SKIP_FILE'                    -- if the file has any parsing error,
                                          -- skip it entirely and continue loading
                                          -- the remaining files — load won't crash
PURGE    = FALSE;                         -- keep files in stage after loading
                                          -- allows reload if something goes wrong
                                          -- Snowflake won't double-load same file
                                          -- unless FORCE = TRUE is added
```

### What the result columns mean

Each COPY INTO returns one row showing what happened:

| Column | Meaning |
|--------|---------|
| `file` | Which file was processed |
| `status` | `LOADED` = success · `SKIPPED` = already loaded previously |
| `rows_parsed` | Total rows read from the file |
| `rows_loaded` | Rows successfully inserted into the table |
| `errors_seen` | Number of rows that failed to parse |

### ON_ERROR options explained

| Option | Behaviour |
|--------|-----------|
| `ABORT_STATEMENT` | Default — any error stops everything immediately |
| `CONTINUE` | Skip bad rows, keep loading good rows |
| `SKIP_FILE` | Skip the entire file if any error is found |
| `SKIP_FILE_n` | Skip file only if errors exceed n rows |

We use `SKIP_FILE` — if one file has a problem the other 8 still load cleanly.

---
## Cell 3 — Load RAW_ORDERS

The central table — one row per order. Every other table joins back to this through `order_id`.
Expected rows: **~99,441**

In [ ]:
COPY INTO RAW_ORDERS
FROM @OLIST_STAGE/olist_orders_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 4 — Load RAW_ORDER_ITEMS

One row per item within an order. More rows than RAW_ORDERS because one order can have multiple items. Different items in the same order can come from different sellers.
Expected rows: **~112,650**

In [ ]:
COPY INTO RAW_ORDER_ITEMS
FROM @OLIST_STAGE/olist_order_items_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 5 — Load RAW_CUSTOMERS

Customer location data. Remember: `customer_id` is per-order, not per person. Use `customer_unique_id` to count distinct customers.
Expected rows: **~99,441**

In [ ]:
COPY INTO RAW_CUSTOMERS
FROM @OLIST_STAGE/olist_customers_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 6 — Load RAW_SELLERS

Small dimension table — one row per seller registered on the platform.
Expected rows: **~3,095**

In [ ]:
COPY INTO RAW_SELLERS
FROM @OLIST_STAGE/olist_sellers_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 7 — Load RAW_PRODUCTS

One row per product. Categories are in Portuguese — joined to RAW_CATEGORY_TRANSLATION in the STAGED layer for English names. Note: column names `product_name_lenght` and `product_description_lenght` have a typo that exists in the original dataset — preserved intentionally in RAW.
Expected rows: **~32,951**

In [ ]:
COPY INTO RAW_PRODUCTS
FROM @OLIST_STAGE/olist_products_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 8 — Load RAW_PAYMENTS

One row per payment event per order. An order can have multiple rows here — e.g. part paid by voucher, rest by credit card. Sum `payment_value` by `order_id` for total order value.
Expected rows: **~103,886**

In [ ]:
COPY INTO RAW_PAYMENTS
FROM @OLIST_STAGE/olist_order_payments_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 9 — Load RAW_REVIEWS

Customer satisfaction data. `review_score` (1–5 stars) is one of the most valuable columns for analytics. Comment fields are often NULL — many customers only leave a score.
Expected rows: **~99,224**

In [ ]:
COPY INTO RAW_REVIEWS
FROM @OLIST_STAGE/olist_order_reviews_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 10 — Load RAW_GEOLOCATION

The largest table — over 1 million rows. Maps Brazilian zip code prefixes to lat/lng coordinates. Has duplicate zip codes with slightly different coordinates — deduplicated in the STAGED layer.
Expected rows: **~1,000,163**

In [ ]:
COPY INTO RAW_GEOLOCATION
FROM @OLIST_STAGE/olist_geolocation_dataset.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 11 — Load RAW_CATEGORY_TRANSLATION

Tiny lookup table — only 71 rows. Maps Portuguese product category names to English. Used in STAGED to add English names to every product.
Expected rows: **~71**

In [ ]:
COPY INTO RAW_CATEGORY_TRANSLATION
FROM @OLIST_STAGE/product_category_name_translation.csv
FILE_FORMAT = (
    FORMAT_NAME                  = 'OLIST_CSV_FORMAT',
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR     = 'SKIP_FILE'
PURGE        = FALSE;

---
## Cell 12 — Verify row counts

Query `INFORMATION_SCHEMA.TABLES` to confirm all 9 tables have the expected number of rows.

### What each result means

- `row_count = expected` → load was successful
- `row_count = 0` → COPY INTO failed or was skipped — check Cell 13 (LOAD_HISTORY)
- `row_count = expected` but status was `SKIPPED` → file was already loaded in a previous run — this is fine, data is already there

> **🔍 Concept Note: Real-time Metadata**
> 
> `INFORMATION_SCHEMA.TABLES` is a real-time system view — no latency, always current. It shows metadata about every table in the database including row count, bytes used, and creation timestamp.

In [ ]:
-- Row counts for all 9 RAW tables
-- Compare actual vs expected to confirm the load was successful
SELECT
    table_name                             AS table_name,
    row_count                              AS actual_rows,
    CASE table_name
        WHEN 'RAW_ORDERS'               THEN 99441
        WHEN 'RAW_ORDER_ITEMS'          THEN 112650
        WHEN 'RAW_CUSTOMERS'            THEN 99441
        WHEN 'RAW_SELLERS'              THEN 3095
        WHEN 'RAW_PRODUCTS'             THEN 32951
        WHEN 'RAW_PAYMENTS'             THEN 103886
        WHEN 'RAW_REVIEWS'              THEN 99224
        WHEN 'RAW_GEOLOCATION'          THEN 1000163
        WHEN 'RAW_CATEGORY_TRANSLATION' THEN 71
    END                                    AS expected_rows,
    CASE
        WHEN row_count = 0 THEN 'EMPTY — load failed'
        WHEN row_count = CASE table_name
            WHEN 'RAW_ORDERS'               THEN 99441
            WHEN 'RAW_ORDER_ITEMS'          THEN 112650
            WHEN 'RAW_CUSTOMERS'            THEN 99441
            WHEN 'RAW_SELLERS'              THEN 3095
            WHEN 'RAW_PRODUCTS'             THEN 32951
            WHEN 'RAW_PAYMENTS'             THEN 103886
            WHEN 'RAW_REVIEWS'              THEN 99224
            WHEN 'RAW_GEOLOCATION'          THEN 1000163
            WHEN 'RAW_CATEGORY_TRANSLATION' THEN 71
        END THEN 'OK — match'
        ELSE 'CHECK — mismatch'
    END                                    AS result
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_RAW'
  AND table_name   LIKE 'RAW_%'
ORDER BY table_name;

---
## Cell 13 — Check LOAD_HISTORY

`INFORMATION_SCHEMA.LOAD_HISTORY` records every `COPY INTO` operation ever run against tables in your database. Each row tells you exactly what happened: which file, how many rows were parsed and loaded, how many errors, and the final status.

### When to use this

- After every load — confirm what actually ran
- When row counts don't match — find out why
- When status was `SKIPPED` — see when the file was originally loaded

> **🔍 Concept Note: LOAD_HISTORY vs COPY_HISTORY**
> 
> | Feature | `INFORMATION_SCHEMA.LOAD_HISTORY` | `SNOWFLAKE.ACCOUNT_USAGE.COPY_HISTORY` |
> |---------|-----------------------------------|----------------------------------------|
> | **Scope** | Current database | Entire account |
> | **Latency** | **Real-time** (Immediate check) | Up to 45 min (Long-term audit) |
> | **Retention** | 14 days | 365 days |

In [ ]:
-- Audit log of all COPY INTO operations on SHOPFLOW_RAW
-- Shows file name, rows loaded, errors, and status for every load
SELECT
    TO_CHAR(LAST_LOAD_TIME, 'YYYY-MM-DD HH24:MI') AS load_time,
    TABLE_NAME,
    ROW_COUNT                                      AS rows_loaded,
    ROW_PARSED                                     AS rows_parsed,
    ERROR_COUNT                                    AS errors,
    STATUS
FROM INFORMATION_SCHEMA.LOAD_HISTORY
WHERE SCHEMA_NAME = 'SHOPFLOW_RAW'
ORDER BY LAST_LOAD_TIME DESC
LIMIT 20;

---
## Cell 14 — Data quality spot checks

Run one check at a time. Each gives you a specific insight about your raw data — useful for designing the STAGED layer transformations in Phase 2.

Run each SELECT independently so you see its full result set.

In [ ]:
-- Check 1: Order status distribution
-- Shows the delivered vs cancelled vs processing split
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM RAW_ORDERS
GROUP BY 1
ORDER BY 2 DESC;

In [ ]:
-- Check 2: Payment type mix
-- Credit card vs boleto vs voucher — and average value per type
SELECT
    payment_type,
    COUNT(*)                            AS payment_count,
    ROUND(AVG(payment_value::FLOAT), 2) AS avg_value_brl
FROM RAW_PAYMENTS
GROUP BY 1
ORDER BY 2 DESC;

In [ ]:
-- Check 3: Review score distribution
-- Customer satisfaction snapshot — 5-star vs 1-star ratio
SELECT
    review_score,
    COUNT(*) AS review_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM RAW_REVIEWS
GROUP BY 1
ORDER BY review_score::INTEGER;

In [ ]:
-- Check 4: NULL check on key columns in RAW_ORDERS
-- Identify data quality issues before STAGED type casting
SELECT
    COUNT(*)                                                          AS total_rows,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END)                AS null_order_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)             AS null_customer_id,
    SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END)            AS null_status,
    SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS null_purchase_ts,
    SUM(CASE WHEN order_delivered_customer_date IS NULL
             THEN 1 ELSE 0 END)                                       AS null_delivery_date
FROM RAW_ORDERS;

In [ ]:
-- Check 5: Top 10 product categories (English)
-- Which categories dominate the catalogue
SELECT
    COALESCE(t.product_category_name_english, 'uncategorised') AS category_english,
    COUNT(*) AS product_count
FROM RAW_PRODUCTS p
LEFT JOIN RAW_CATEGORY_TRANSLATION t
       ON p.product_category_name = t.product_category_name
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10;

---
## Cell 15 — Suspend warehouse

Manually suspend the warehouse when you are done. Even though `SHOPFLOW_WH` has `AUTO_SUSPEND = 60`, suspending explicitly stops the billing clock immediately rather than waiting 60 more seconds.

> **💸 Cost Optimization Tip: Warehouse Billing**
> 
> Snowflake warehouses are billed in **60-second minimum increments** every time they start. Suspending immediately after your last query explicitly stops the clock and eliminates that potential 60 seconds of idle time cost.

In [ ]:
-- Suspend the warehouse — stop the billing clock immediately
ALTER WAREHOUSE SHOPFLOW_WH SUSPEND;

In [ ]:
-- Confirm warehouse is now suspended
SHOW WAREHOUSES LIKE 'SHOPFLOW%';

---
## 🛠 Self-Guided Exploration: "Trust but Verify"

Now that you've executed the COPY INTO statements, let's test your understanding of data loading and metadata in Snowflake. Try to write these queries from scratch!

### Challenge 1: Filter the Stage
**Objective:** Practice using the `LIST` command with pattern matching.
**Task:** Write a `LIST` command that only returns files in `@OLIST_STAGE` containing the word `payments`.

In [ ]:
-- Write your SQL for Challenge 1 here


### Challenge 2: Audit the Load History
**Objective:** Master the `LOAD_HISTORY` view.
**Task:** Write a query against `INFORMATION_SCHEMA.LOAD_HISTORY` to find only the loads that happened for the `RAW_REVIEWS` table.

In [ ]:
-- Write your SQL for Challenge 2 here


### Challenge 3: Table Size Check
**Objective:** Inspect table metadata.
**Task:** Query `INFORMATION_SCHEMA.TABLES` to find the exact `bytes` consumed by the `RAW_PRODUCTS` table.

In [ ]:
-- Write your SQL for Challenge 3 here


### Challenge 4: Quick Data Profiling
**Objective:** Verify raw data contents.
**Task:** Write a `SELECT` query on `RAW_ORDERS` to count how many orders have an `order_status` of `'canceled'`.

In [ ]:
-- Write your SQL for Challenge 4 here


---
## Phase 1 complete

If Cell 12 shows all 9 tables with matching row counts — your RAW layer is fully loaded.

### What you just did

```text
Kaggle CSV files
      │  (uploaded via Snowsight UI)
      ▼
OLIST_STAGE  (internal stage)
      │  (COPY INTO — this notebook)
      ▼
SHOPFLOW_RAW  ← you are here
  RAW_ORDERS · RAW_ORDER_ITEMS · RAW_CUSTOMERS
  RAW_SELLERS · RAW_PRODUCTS · RAW_PAYMENTS
  RAW_REVIEWS · RAW_GEOLOCATION · RAW_CATEGORY_TRANSLATION
```

### Key Snowflake concepts you practised

| Topic | Covered |
|-------|--------|
| COPY INTO syntax | ✅ |
| FILE_FORMAT — FORMAT_NAME | ✅ |
| ON_ERROR options | ✅ |
| PURGE option | ✅ |
| COPY INTO idempotency | ✅ |
| LIST stage | ✅ |
| INFORMATION_SCHEMA.LOAD_HISTORY | ✅ |
| INFORMATION_SCHEMA.TABLES | ✅ |
| Warehouse suspend | ✅ |

### Next — Phase 2: STAGED layer

- Cast all VARCHAR columns to proper data types (TIMESTAMP_NTZ, NUMBER, FLOAT)
- Deduplicate rows using window functions
- Handle NULLs explicitly
- Fix column name typos from RAW (e.g. `lenght` → `length`)
- Validate referential integrity across tables
- Automate with Streams + Tasks